In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve().parent
GROUND = ROOT / "ground_stations"
SAT = ROOT / "sat_data"

In [ ]:
import pandas as pd
import rasterio as rio
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pyproj import Transformer
from sklearn.metrics import r2_score

In [ ]:
# Cargar GeoTIFF y coordenadas de estaciones

with rio.open(SAT / "precip_1980_2024_stack.tif") as src:
    chirps = src.read()   # (B, H, W)
    chirps_crs = src.crs
    chirps_nodata = src.nodata
    n_bands = src.count
    
st = pd.read_csv(GROUND / "stations_lat_lon.csv", encoding='latin1')

In [ ]:
# Transformar coords a CRS del raster
to_raster = Transformer.from_crs("EPSG:4326", chirps_crs, always_xy=True)
st["x"], st["y"] = to_raster.transform(st["lon"].values, st["lat"].values)

### El objeto transformer traduce desde 4326 al CRS destino (chirps_crs), es para asegurarse de que las estaciones y el raster estén en el mismo

In [ ]:
# Generar un datetime con las fechas
dates = pd.date_range("1980-01-01", periods=n_bands, freq="MS")

In [ ]:
with rio.open(SAT / "precip_1980_2024_stack.tif") as src:
    # sample espera iterable de (x,y) en el CRS del raster
    samples = list(src.sample(list(zip(st["x"], st["y"]))))  # len = n_stations, cada uno shape (n_bands,)
    arr = np.vstack(samples)  # (n_stations, n_bands)

In [ ]:
# Reemplazar NoData por NaN
if chirps_nodata is not None:
    arr = np.where(arr == raster_nodata, np.nan, arr)

In [ ]:
# Armar DataFrame largo: una fila por estación-fecha
df = (
    pd.DataFrame(arr, index=st["station_id"], columns=dates)
      .rename_axis(index="station_id", columns="date")
      .stack()
      .reset_index(name="precip_mm")
)

df

In [ ]:
# Plotear serie temporal chirps de todas las estaciones
# Hay que tirarle una onda, alargarlos y ponerles el título de cada estación

for station, subdf in df.groupby('station_id'):
    plt.figure()
    plt.plot(subdf['date'], subdf['precip_mm'])
    plt.show()

In [ ]:
# Cargar datos medidos, ya pasados por limpieza
p_medida = pd.read_csv(GROUND / 'p_mensual_completo.csv')
p_medida['fecha'] = pd.to_datetime(p_medida['fecha'], errors='coerce')
p_medida.info()

## FUNCIÓN PARA GENERAR UN DF CON PARES MEDIDO-SATELITAL POR ESTACIÓN

In [ ]:
# Armar dicts de configuración de datos satelitales y medidos

sat_cfg = {
    "df": df,
    "station": "station_id",
    "date": "date",
    "precip": "precip_mm"
}

ground_cfg = {
    "df": p_medida,
    "station": "estacion",
    "date": "fecha",
    "precip": "p_mensual"
}


def compare_sat_ground(station_name, sat_cfg, ground_cfg):
    df_sat = sat_cfg["df"]
    df_ground = ground_cfg["df"]

    col_sat_station = sat_cfg["station"]
    col_sat_date    = sat_cfg["date"]
    col_sat_p       = sat_cfg["precip"]

    col_ground_station = ground_cfg["station"]
    col_ground_date    = ground_cfg["date"]
    col_ground_p       = ground_cfg["precip"]
    
    df = (
        df_sat[df_sat[col_sat_station] == station_name]
          .merge(
              df_ground[df_ground[col_ground_station] == station_name],
              left_on=[col_sat_station, col_sat_date],
              right_on=[col_ground_station, col_ground_date],
              how="left"
          )
    )
    
    df = df.rename(
        columns={
            col_sat_p: "p_sat",
            col_ground_p: "p_ground"
        }
    )
    
    series_start = df.loc[df["p_ground"].notna(), col_ground_date].min()
    series_end   = df.loc[df["p_ground"].notna(), col_ground_date].max()
    df = df.loc[df[col_sat_date].between(series_start, series_end)]
    
    df = df.drop(columns=[col_ground_date, col_ground_station])

    print(df.head())

    fig, ax = plt.subplots(figsize=(16,5))
    ax.plot(df['date'], df['p_sat'], label='CHIRPS')
    ax.plot(df['date'], df['p_ground'], label='Estación')
    ax.legend()
    ax.set_title(f'Estación {station_name}')
    ax.set_ylabel("P mensual (mm)")
    plt.show()
    
    return df

df_lima = compare_sat_ground("lima", sat_cfg, ground_cfg)

In [ ]:
# Función para graficar scatter con ajuste lineal

def scatter_fit(station_name, df):
    # Filtrar NaN y extraer series
    mask = df[["p_ground","p_sat"]].dropna()
    x = mask["p_ground"].values
    y = mask["p_sat"].values
    
    fig, ax = plt.subplots(figsize=(7,7))
    
    # Scatter
    ax.scatter(x, y, alpha=0.7)
    
    # Rango común
    min_val = min(x.min(), y.min())
    max_val = max(x.max(), y.max()) + 10
    
    # Línea 1:1
    ax.plot([min_val, max_val], [min_val, max_val], color="black",
            linestyle="--", linewidth=0.6, label="1:1")
    
    # Ajuste lineal (y = m x + b)
    m, b = np.polyfit(x, y, 1)
    x_fit = np.array([min_val, max_val])
    y_fit = m * x_fit + b
    ax.plot(x_fit, y_fit, color="red", linewidth=1.5, label=f"Ajuste lineal (m={m:.2f}, b={b:.2f})")
    
    # Ejes iguales y formato
    ax.set_xlim(min_val, max_val)
    ax.set_ylim(min_val, max_val)
    ax.set_aspect("equal")
    ax.set_title(f"Estación {station_name} medidos vs. satelitales")
    ax.set_xlabel("Precipitación medida (mm/mes)")
    ax.set_ylabel("Precipitación CHIRPS (mm/mes)")
    ax.legend()
    plt.show()

scatter_fit("Lima", df_lima)

In [ ]:
# Función para calcular las métricas

def compute_metrics(x, y):
    # convertir a arrays
    x = np.asarray(x)
    y = np.asarray(y)

    # máscara: valores finitos y no NaN en ambos
    mask = ~(np.isnan(x) | np.isnan(y) | np.isinf(x) | np.isinf(y))
    x_valid = x[mask]
    y_valid = y[mask]

    # r2 de sklearn
    r2_sk = r2_score(x_valid, y_valid)

    # correlación de Pearson y su cuadrado
    r = np.corrcoef(x_valid, y_valid)[0,1]
    r2_pearson2 = r**2

    return {"r2_sk": r2_sk, "r2_pearson2": r2_pearson2}

metrics = compute_metrics(df_lima["p_ground"], df_lima["p_sat"])
print(metrics)